# Pair and morph two snapshots

`morph` turns one snapshot into another, and `pair` decides which partials go
together. A snapshot holds the peaks of one analysis frame, so a morph glides
from one frozen frame to another: how each sound changes over time (the next
syllable of the speech, the decay of the conga) is not part of it. This
notebook morphs `speech1` (41 peaks) into `conga3` (10 peaks) with the four
pairings, plays each one, shows curved glides and writes one morph as an events
table. Read `visual_tsv` first for snapshots.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import audiospylt as asp

## Source and target

Each table holds the peaks of one DFT frame. Speech reaches 3055.3 Hz; the
conga stays at or below 600.0 Hz.

In [ ]:
speech = asp.read_table("../data/tables/speech1.tsv")
conga = asp.read_table("../data/tables/conga3.tsv")
print(f"speech1 {len(speech)} peaks, {speech.freq.min():.1f}–{speech.freq.max():.1f} Hz")
print(f"conga3 {len(conga)} peaks, {conga.freq.min():.1f}–{conga.freq.max():.1f} Hz")
asp.nb.show(asp.viz.plot_snapshot(speech, conga, labels=["speech1", "conga3"], freq_scale="log"))

## Listen to the two frames

`snapshot_to_events` holds every peak of a snapshot at its frequency and
amplitude over a time span, here 2.0 s: the sound a morph starts from and the
sound it glides toward. Listen for two steady sounds with no attack and no
decay. The renders are far below full scale, so `nb.play` normalizes each one.

In [ ]:
sr = 44100
for label, snap in (("speech1", speech), ("conga3", conga)):
    audio = asp.synthesize(asp.snapshot_to_events(snap, 0.0, 2.0), sr=sr)
    mean = np.average(snap.freq, weights=snap.amp)
    print(f"{label}: peak {np.max(np.abs(audio)):.4f}, amplitude-weighted mean {mean:.1f} Hz")
    asp.nb.play(audio, sr, normalize=True)
print(f"closest speech peaks: {np.diff(np.sort(speech.freq)).min():.2f} Hz apart")

Normalized, the two play equally loud, but the conga frame is much quieter
(peak 0.0097 against 0.0798) and leans lower: its amplitude-weighted mean
frequency is 335.4 Hz, against 510.8 Hz for the speech. The speech frame
wavers because some of its peaks lie 5.26 Hz apart, and two sinusoids that
close beat 5.26 times per second. Every morph below starts on the speech frame
and glides toward the conga frame.

## Equal pairing

With 41 speech peaks and 10 conga peaks, several speech peaks share a conga
peak. `"equal"`, `"power"` and `"sigmoid"`, the maps of earlier AudioSpylt
versions, keep both snapshots in order, low to high, and differ in how many
speech peaks each conga peak receives; `"equal"` spreads them evenly. `plot_morph_map` draws each pair as a
line from source (left) to target (right); `morph` turns the pairs into glides
over 2.0 s. Listen for the whole speech spectrum sinking into the conga's range.

In [ ]:
pairing = asp.pair(speech, conga, method="equal")
print("speech peaks per conga peak:", pairing.groupby("dst").size().tolist())
asp.nb.show(asp.viz.plot_morph_map(pairing))
morphed = asp.morph(speech, conga, 2.0, method="equal")
asp.nb.show(asp.viz.plot_events(morphed))
asp.nb.play(asp.synthesize(morphed, sr=sr), sr, normalize=True)

Each conga peak receives 3 to 5 speech peaks. The lowest speech peak, 97.4 Hz,
glides down to 31.2 Hz and the highest, 3055.3 Hz, falls to 600.0 Hz.

## Duplicates: full or primary

Each conga peak therefore ends as 3 to 5 glides. The default for equal, power
and sigmoid, `duplicates="full"`, gives every one of them the conga peak's full
amplitude, as earlier versions did; `duplicates="primary"` keeps it on one glide per conga peak and
fades the others to 0, so the morph arrives at the conga frame's own level and
balance. `help(asp.morph)` explains the two rules. Listen for the quieter
arrival of the primary morph.

In [ ]:
primary = asp.morph(speech, conga, 2.0, method="equal", duplicates="primary")
for label, events in (("full", morphed), ("primary", primary)):
    mean = np.average(events.freq_stop, weights=events.amp_stop)
    print(f"{label}: {(events.amp_stop == 0).sum()} of {len(events)} glides end at amp 0, "
          f"end mean {mean:.1f} Hz")
    asp.nb.play(asp.synthesize(events, sr=sr), sr, normalize=True)

With `"primary"`, 31 of the 41 glides fade out (the extra copies onto the 10
conga peaks), and the end has the conga frame's own amplitude-weighted mean,
335.4 Hz. With `"full"` all 41 arrive at full amplitude, so each conga peak
sounds 3 to 5 times over and the ending stays as loud as the start.

## Power pairing

`"power"` bends the equal map with an exponent: below 1 it crowds the speech
peaks onto the low conga peaks, above 1 toward the high ones.
`parameter_range` returns the range in which every conga peak still gets at
least one speech peak; without `power=`, `pair` and `morph` use the middle of
the range. The cell draws the maps at both ends.

In [ ]:
lo, hi = asp.parameter_range(speech, conga, "power")
for power in (lo, hi):
    p = asp.pair(speech, conga, method="power", power=power)
    print(f"power={power:.4f}: speech peaks per conga peak {p.groupby('dst').size().tolist()}")
    asp.nb.show(asp.viz.plot_morph_map(p))
default = asp.pair(speech, conga, method="power").attrs["morph"]["parameter"]
print(f"default power={default:.4f}")

At power=0.2032, 13 and 15 of the 41 speech peaks go to the two lowest conga
peaks; at 1.8964 the counts grow toward the top, up to 8 on the second-highest
peak. The default, 1.0498, is close to equal. With `duplicates="full"`, a conga
peak that receives more glides also ends louder, so the exponent tilts the
ending: listen for a dark ending at the low end of the range and a brighter one
at the high end.

In [ ]:
for power in (lo, hi):
    m = asp.morph(speech, conga, 2.0, method="power", power=power)
    mean = np.average(m.freq_stop, weights=m.amp_stop)
    print(f"power={power:.4f}: end mean {mean:.1f} Hz")
    asp.nb.show(asp.viz.plot_events(m))
    asp.nb.play(asp.synthesize(m, sr=sr), sr, normalize=True)

The amplitude-weighted mean of the end points is 187.2 Hz at the low end and
393.3 Hz at the high end, either side of the conga frame's own 335.4 Hz.

## Sigmoid pairing

`"sigmoid"` bends the map with an S curve, which crowds the speech peaks onto
the middle conga peaks; `sigmoid_range` sets how steep the S is. The cell draws
the maps at both ends of its range.

In [ ]:
lo, hi = asp.parameter_range(speech, conga, "sigmoid")
for sigmoid_range in (lo, hi):
    p = asp.pair(speech, conga, method="sigmoid", sigmoid_range=sigmoid_range)
    print(f"sigmoid_range={sigmoid_range:g}: speech peaks per conga peak {p.groupby('dst').size().tolist()}")
    asp.nb.show(asp.viz.plot_morph_map(p))
default = asp.pair(speech, conga, method="sigmoid").attrs["morph"]["parameter"]
print(f"default sigmoid_range={default:g}")

At 0.01 the map is close to `"equal"` (3 to 5 speech peaks per conga peak); at
5.53 the middle conga peaks take 10 and 11 speech peaks, and the two lowest and
two highest take 1 each. The default is 2.77. Listen for the glides gathering
in the middle of the conga range, more tightly at the high end of the range.

In [ ]:
for sigmoid_range in (default, hi):
    m = asp.morph(speech, conga, 2.0, method="sigmoid", sigmoid_range=sigmoid_range)
    print(f"sigmoid_range={sigmoid_range:g}: glides per conga peak "
          f"{m.groupby('freq_stop').size().tolist()}")
    asp.nb.show(asp.viz.plot_events(m))
    asp.nb.play(asp.synthesize(m, sr=sr), sr, normalize=True)

At the default, 2.77, the middle conga peaks take 6 or 7 glides and the outer
ones 2; at 5.53 the middle takes up to 11 and the outer peaks 1 each.

## Outside the range

Past the ends of the range some conga peaks get no partner: `pair` warns, and a
morph ends on the peaks that are used. A negative parameter flips the map, so
the lowest speech peaks go to the highest conga peaks. Before you run the cell:
how many conga peaks does `power=50.5` use?

In [ ]:
for method, kw in [("power", {"power": 50.5}), ("sigmoid", {"sigmoid_range": 300}),
                   ("power", {"power": -1.0})]:
    p = asp.pair(speech, conga, method=method, **kw)
    print(f"{kw}: {p.dst.nunique()} of {len(conga)} conga peaks used; "
          f"lowest speech peak → {p.freq_stop.iloc[0]:.1f} Hz")

`power=50.5` uses 3 of the 10 conga peaks and `sigmoid_range=300` uses 4; the
warnings count the unused ones. `power=-1.0` uses all 10, so it does not warn,
but it sends the lowest speech peak, 97.4 Hz, to 600.0 Hz, the top of `conga3`.
Listen to two of these as morphs: `power=50.5` piles the glides onto the top of
the conga range, and `power=-1.0` turns the map upside down, so the low speech
peaks rise while the high ones fall and the glides cross.

In [ ]:
for power in (50.5, -1.0):
    m = asp.morph(speech, conga, 2.0, method="power", power=power)
    print(f"power={power}: {(m.freq_stop > m.freq_start).sum()} of {len(m)} glides rise; "
          f"per conga peak {m.groupby('freq_stop').size().to_dict()}")
    asp.nb.show(asp.viz.plot_events(m))
    asp.nb.play(asp.synthesize(m, sr=sr), sr, normalize=True)

At `power=50.5`, 20 glides end on 546.25 Hz and 20 on 600.0 Hz; with
`duplicates="full"` the morph ends on a loud two-note chord over one quiet low
peak at 31.25 Hz. At `power=-1.0`, 18 of the 41 glides rise.

## Transport

`"transport"` (new in v1) treats the amplitudes as mass and moves the speech
spectrum onto the conga spectrum, low to high, with the least movement in
frequency. A speech peak can split between two conga peaks, so `pair` can
return more rows than either snapshot has. Before you run the cell: how many
rows does it return here?

In [ ]:
pt = asp.pair(speech, conga, method="transport")
mt = asp.morph(speech, conga, 2.0, method="transport")
print(f"pair {len(pt)} rows, morph {len(mt)} events")
asp.nb.show(asp.viz.plot_morph_map(pt))
asp.nb.show(asp.viz.plot_events_3d(mt))

50 pair rows, 41 events. By default a transport morph keeps a row only where it
carries a partial's amplitude at the start or at the end
(`duplicates="primary"`), so both ends sound at the snapshots' levels.
`help(asp.pair)` lists `weight`, the other transport option.

## Transport against equal

The primary equal morph and the transport morph start and end on the same two
frames; only the paths between them differ. Listen for the shorter glides of
the transport morph: the loud speech peaks land on nearer conga peaks instead
of the ones their rank order assigns. The cell measures each morph's mean
glide, weighted by the speech amplitudes.

In [ ]:
for label, events in (("equal primary", primary), ("transport", mt)):
    cents = np.abs(1200 * np.log2(events.freq_stop / events.freq_start))
    print(f"{label}: mean glide {np.average(cents, weights=events.amp_start):.0f} cents")
    asp.nb.play(asp.synthesize(events, sr=sr), sr, normalize=True)

Weighted by the speech amplitudes, the equal glides cover 1258 cents on average
and the transport glides 662, about half as far.

## Curved frequency glides

By default each glide is linear in Hz and in amplitude, one row per pair.
`freq_space="pitch"` moves in log frequency (constant cents per second);
`freq_curve` bends that path with the Csound `transeg` / SuperCollider `Env`
curve: 0 is linear, > 0 starts slowly and arrives fast, < 0 the reverse. One
partial from 220.0 Hz at amplitude 0.5 to 880.0 Hz at 0.05 over 2.0 s makes
the three shapes easy to read. Before you run the cell: at 1.0 s, which glide
sits lowest — linear Hz, pitch, or pitch with `freq_curve=4`?

In [ ]:
partial = pd.DataFrame({"freq": [220.0], "amp": [0.5]})
target = pd.DataFrame({"freq": [880.0], "amp": [0.05]})

def freq_at(events, t=1.0):
    row = events[(events.time_start <= t) & (events.time_stop >= t)].iloc[0]
    u = (t - row.time_start) / (row.time_stop - row.time_start)
    return float(row.freq_start + u * (row.freq_stop - row.freq_start))

hz = asp.morph(partial, target, 2.0)
pitch = asp.morph(partial, target, 2.0, freq_space="pitch")
late = asp.morph(partial, target, 2.0, freq_space="pitch", freq_curve=4)
for label, e in (("hz", hz), ("pitch", pitch), ("pitch curve=4", late)):
    print(f"{label}: {len(e)} rows, {freq_at(e):.1f} Hz at 1.0 s")
asp.nb.show(asp.viz.plot_events(hz, pitch, late, labels=["hz", "pitch", "pitch curve=4"],
                                freq_scale="log"))

`freq_curve=4` sits lowest. The linear Hz morph is 1 row and reads 550.0 Hz at
1.0 s, the midpoint in hertz. `freq_space="pitch"` keeps a steady rate in
cents, so at 1.0 s it prints 440.2 Hz (21 rows); with `freq_curve=4` it holds
back and reaches only 259.7 Hz (38 rows). The pitch glide prints a little above
440 Hz because the rows are straight lines in Hz within `max_cents` (1 cent by
default); the exact curve passes 440 Hz, the geometric mean of 220.0 and
880.0 Hz. On the log frequency axis the pitch glide is a straight line; the Hz
glide bows upward, reaching 440 Hz after a third of its 2.0 s.

## Listen to the three glides

Listen for the shape of each 2.0 s glide: the Hz morph covers its first octave
faster than its second, so it seems to slow down; the pitch morph moves evenly;
`freq_curve=4` holds back and then rushes.

In [ ]:
for label, events in (("hz", hz), ("pitch", pitch), ("pitch curve=4", late)):
    asp.nb.play(asp.synthesize(events, sr=sr), sr, normalize=True)

## Curved amplitude

The same partial falls from amplitude 0.5 to 0.05. `amp_curve` uses the same
sign rule as frequency: > 0 moves late, < 0 early. The cell compares the
default with `amp_curve=-4` (early fall) and `amp_curve=4` (late fall), and
prints the level at 1.0 s in dB.

In [ ]:
def amp_at(events, t=1.0):
    row = events[(events.time_start <= t) & (events.time_stop >= t)].iloc[0]
    u = (t - row.time_start) / (row.time_stop - row.time_start)
    return float(row.amp_start + u * (row.amp_stop - row.amp_start))

for c in (0, -4, 4):
    e = asp.morph(partial, target, 2.0, amp_curve=c)
    a = amp_at(e)
    print(f"amp_curve={c}: {len(e)} rows, {a:.4f} ({asp.amplitude_to_db(a):.1f} dB) at 1.0 s")

At 1.0 s the default is still 0.2750 (−11.2 dB). `amp_curve=-4` has already
fallen to 0.1038 (−19.7 dB); `amp_curve=4` is still at 0.4462 (−7.0 dB). Each
curved morph uses 19 rows; the linear one stays 1 row.

## Curved transport morph

The docs example bends the speech1 → conga3 transport morph in pitch space with
`freq_curve=4` and `amp_curve=-2`. A coarser `max_cents` keeps the same curve
with fewer linear rows. `synthesize` defaults to `phase="continuous"`, which
renders the chained rows as one glide per pair (the other phase modes click at
every join). `help(asp.morph)` lists the rest. Listen for the curved morph
changing its levels early (`amp_curve=-2`) while its partials stay near the
speech pitches and slide onto the conga partials late (`freq_curve=4`).

In [ ]:
mc = asp.morph(speech, conga, 2.0, method="transport",
               freq_space="pitch", freq_curve=4, amp_curve=-2)
mc10 = asp.morph(speech, conga, 2.0, method="transport",
                 freq_space="pitch", freq_curve=4, amp_curve=-2, max_cents=10)
print(f"linear {len(mt)} events; curved {len(mc)} rows; max_cents=10 → {len(mc10)} rows")
asp.nb.show(asp.viz.plot_events(mc, freq_scale="log"))
for label, events in (("linear", mt), ("curved", mc)):
    audio = asp.synthesize(events, sr=sr)
    print(f"{label}: peak {np.max(np.abs(audio)):.4f}")
    asp.nb.play(audio, sr, normalize=True)

The linear transport morph stays at 41 events; the curved one expands to 844
rows at the default `max_cents`, and to 281 rows with `max_cents=10`. The
render peaks near 0.059, so `nb.play` normalizes it as before.

## Write one morph

`write_table` saves the equal morph as a v1 events table under `tsv/`.

In [ ]:
Path("tsv").mkdir(exist_ok=True)
path = asp.write_table(morphed, "tsv/speech_to_conga_equal.tsv")
print(path, "|", path.read_text().splitlines()[0])